# 04 · Results — the validity/usable gap, drawn

This is the argument in four pictures. Each is regenerated here from the CSVs the
experiments wrote, so nothing below re-runs a method or touches a black box.

1. **The gap** — every method flips the box; far fewer give *usable* recourse.
2. **The funnel** — which of the three audits each method loses people to.
3. **The cost of validity** — what closing the gap costs in effort.
4. **The ablation** — which CAC term is doing the closing.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
ROOT = _here if (_here / "src").exists() else _here.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

RESULTS = ROOT / "results"

def show_image(path):
    """Display a saved figure inline in a notebook; no-op-print in a script."""
    path = pathlib.Path(path)
    try:
        from IPython.display import Image, display  # noqa: PLC0415
        display(Image(filename=str(path)))
    except Exception:
        print("figure saved at", path)

def read_result(name):
    """Read a results CSV, or say which experiment writes it."""
    p = RESULTS / f"{name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"{p} missing - run the experiment that writes it")
    return pd.read_csv(p)

In [ ]:
from src.viz.plot_results import plot_all_results

figs = plot_all_results()
for name in ["usable_gap", "recourse_funnel", "cost_of_validity", "ablation"]:
    if name in figs:
        show_image(figs[name][0])

## The advice itself, one applicant at a time

Rates cannot show what a plan *is*. The recourse-plan figure takes the flagship
applicant — chosen by a rule fixed in advance, blind to the numbers it will draw —
and lays every method's plan side by side: solid bars for what a method asks,
faint hatched bars for what the SCM propagates from it, and a red ring where an
audit objected.

In [ ]:
from src.viz.plot_counterfactuals import plot_all_counterfactuals, default_flagship

cases = read_result("qualitative_cases")
print("flagship applicant:", default_flagship(cases))
plans = plot_all_counterfactuals(cases)
show_image(plans["flagship"][0])

In [ ]:
# The clearest contrast: a HELOC applicant where the baselines fail the SCM audit
# while CAC stays usable. (Skips cleanly if that case was not in this run.)
key = "heloc_side_effect"
if key in plans:
    show_image(plans[key][0])
else:
    print(key, "not produced in this run; available:", sorted(plans))

## What to take away

- Flipping the model is **not** the same as giving usable recourse — the gap is
  large and it is the SCM audit that opens most of it.
- CAC closes the gap, and the ablation shows the **causal term** is what does it:
  remove it and CAC's usable rate collapses toward the baselines'.
- CAC is not free — it typically asks for **more effort** — but it is effort spent
  on a plan that will actually come true. That trade is the paper's claim.